## 4. Modeling

In this section, a machine learning model was trained using the features obtained in the previous step.
The data was split into training and test sets, and classification was performed using the Logistic Regression algorithm.
Model performance was evaluated using accuracy, classification report, and confusion matrix.


In [51]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [52]:
X = joblib.load("X_features.pkl")
y = joblib.load("y_labels.pkl")

In [53]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=500,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape[0], "Test:", X_test.shape[0])

Train: 1594 Test: 500


In [54]:
# 3) CV setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [55]:
# Pipelines 
pipelines = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LogisticRegression(
            solver="liblinear",
            max_iter=5000,
            random_state=42
        ))
    ]),
    "Linear SVM": Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LinearSVC(
            C=1.0,
            max_iter=50000
        ))
    ]),
    "Random Forest": Pipeline([
    ("clf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
]),

    "MLP": Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", MLPClassifier(
            hidden_layer_sizes=(128,),
            max_iter=200,
            early_stopping=True,
            n_iter_no_change=10,
            random_state=42
        ))
    ])
}

In [56]:
# Train/Test evaluation helper
rows = []

def eval_model(name, pipe, X_train, X_test, y_train, y_test, feature_set, rows):
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

    rows.append({
        "Feature": feature_set,
        "Model": name,
        "Accuracy": acc,
        "Precision (w)": report["weighted avg"]["precision"],
        "Recall (w)": report["weighted avg"]["recall"],
        "F1-score (w)": report["weighted avg"]["f1-score"]
    })

    print("\n===", feature_set, "|", name, "===")
    print("Accuracy:", acc)
    print(classification_report(y_test, y_pred, zero_division=0))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

for name, pipe in pipelines.items():
    eval_model(name, pipe, X_train, X_test, y_train, y_test, "TFIDF+Custom", rows)

df_results = pd.DataFrame(rows).sort_values("F1-score (w)", ascending=False)
display(df_results)


=== TFIDF+Custom | Logistic Regression ===
Accuracy: 0.824
              precision    recall  f1-score   support

           0       0.79      0.79      0.79       211
           1       0.85      0.85      0.85       289

    accuracy                           0.82       500
   macro avg       0.82      0.82      0.82       500
weighted avg       0.82      0.82      0.82       500

Confusion Matrix:
 [[166  45]
 [ 43 246]]

=== TFIDF+Custom | Linear SVM ===
Accuracy: 0.778
              precision    recall  f1-score   support

           0       0.75      0.72      0.73       211
           1       0.80      0.82      0.81       289

    accuracy                           0.78       500
   macro avg       0.77      0.77      0.77       500
weighted avg       0.78      0.78      0.78       500

Confusion Matrix:
 [[152  59]
 [ 52 237]]

=== TFIDF+Custom | Random Forest ===
Accuracy: 0.864
              precision    recall  f1-score   support

           0       0.94      0.73      0.8

,Feature,Model,Accuracy,Precision (w),Recall (w),F1-score (w)
2,TFIDF+Custom,Random Forest,0.864,0.874633,0.864,0.860487
3,TFIDF+Custom,MLP,0.852,0.851637,0.852,0.851693
0,TFIDF+Custom,Logistic Regression,0.824,0.823796,0.824,0.823884
1,TFIDF+Custom,Linear SVM,0.778,0.777222,0.778,0.777456


In [57]:
X_bow = joblib.load("X_bow.pkl")
y_bow = joblib.load("y_labels.pkl")

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    X_bow,
    y_bow,
    test_size=500,
    random_state=42,
    stratify=y_bow
)

for name, pipe in pipelines.items():
    eval_model(name, pipe, Xb_train, Xb_test, yb_train, yb_test, "BoW", rows)

df_results = pd.DataFrame(rows).sort_values(
    ["Feature", "F1-score (w)"],
    ascending=[True, False]
)

display(df_results)



=== BoW | Logistic Regression ===
Accuracy: 0.828
              precision    recall  f1-score   support

           0       0.80      0.80      0.80       211
           1       0.85      0.85      0.85       289

    accuracy                           0.83       500
   macro avg       0.82      0.82      0.82       500
weighted avg       0.83      0.83      0.83       500

Confusion Matrix:
 [[168  43]
 [ 43 246]]

=== BoW | Linear SVM ===
Accuracy: 0.804
              precision    recall  f1-score   support

           0       0.78      0.74      0.76       211
           1       0.82      0.85      0.83       289

    accuracy                           0.80       500
   macro avg       0.80      0.80      0.80       500
weighted avg       0.80      0.80      0.80       500

Confusion Matrix:
 [[157  54]
 [ 44 245]]

=== BoW | Random Forest ===
Accuracy: 0.848
              precision    recall  f1-score   support

           0       0.90      0.72      0.80       211
           1   

,Feature,Model,Accuracy,Precision (w),Recall (w),F1-score (w)
7,BoW,MLP,0.870,0.869923,0.870,0.869958
6,BoW,Random Forest,0.848,0.855425,0.848,0.844530
4,BoW,Logistic Regression,0.828,0.828000,0.828,0.828000
5,BoW,Linear SVM,0.804,0.803234,0.804,0.803288
2,TFIDF+Custom,Random Forest,0.864,0.874633,0.864,0.860487
3,TFIDF+Custom,MLP,0.852,0.851637,0.852,0.851693
0,TFIDF+Custom,Logistic Regression,0.824,0.823796,0.824,0.823884
1,TFIDF+Custom,Linear SVM,0.778,0.777222,0.778,0.777456


In [58]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

X_w2v = np.load("X_w2v.npy")
y_w2v = joblib.load("y_labels.pkl")

Xw_train, Xw_test, yw_train, yw_test = train_test_split(
    X_w2v,
    y_w2v,
    test_size=500,
    random_state=42,
    stratify=y_w2v
)

pipelines_w2v = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(solver="liblinear", max_iter=5000, random_state=42))
    ]),
    "MLP": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", MLPClassifier(
            hidden_layer_sizes=(128,),
            max_iter=200,
            early_stopping=True,
            n_iter_no_change=10,
            random_state=42
        ))
    ])
}

for name, pipe in pipelines_w2v.items():
    eval_model(name, pipe, Xw_train, Xw_test, yw_train, yw_test, "Word2Vec", rows)

df_results = pd.DataFrame(rows).sort_values(
    ["Feature", "F1-score (w)"],
    ascending=[True, False]
)

display(df_results)



=== Word2Vec | Logistic Regression ===
Accuracy: 0.84
              precision    recall  f1-score   support

           0       0.81      0.81      0.81       211
           1       0.86      0.86      0.86       289

    accuracy                           0.84       500
   macro avg       0.84      0.84      0.84       500
weighted avg       0.84      0.84      0.84       500

Confusion Matrix:
 [[171  40]
 [ 40 249]]

=== Word2Vec | MLP ===
Accuracy: 0.81
              precision    recall  f1-score   support

           0       0.78      0.76      0.77       211
           1       0.83      0.85      0.84       289

    accuracy                           0.81       500
   macro avg       0.81      0.80      0.80       500
weighted avg       0.81      0.81      0.81       500

Confusion Matrix:
 [[160  51]
 [ 44 245]]


,Feature,Model,Accuracy,Precision (w),Recall (w),F1-score (w)
7,BoW,MLP,0.870,0.869923,0.870,0.869958
6,BoW,Random Forest,0.848,0.855425,0.848,0.844530
4,BoW,Logistic Regression,0.828,0.828000,0.828,0.828000
5,BoW,Linear SVM,0.804,0.803234,0.804,0.803288
2,TFIDF+Custom,Random Forest,0.864,0.874633,0.864,0.860487
3,TFIDF+Custom,MLP,0.852,0.851637,0.852,0.851693
0,TFIDF+Custom,Logistic Regression,0.824,0.823796,0.824,0.823884
1,TFIDF+Custom,Linear SVM,0.778,0.777222,0.778,0.777456
8,Word2Vec,Logistic Regression,0.840,0.840000,0.840,0.840000
9,Word2Vec,MLP,0.810,0.809393,0.810,0.809534


In [59]:
# Cross-validation
print("\n--- 5-fold CV (Accuracy mean±std) [TRAIN ONLY] ---")
for name, pipe in pipelines.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1)
    print(f"{name}: {scores.mean():.4f} ± {scores.std():.4f}")


--- 5-fold CV (Accuracy mean±std) [TRAIN ONLY] ---
Logistic Regression: 0.8162 ± 0.0111
Linear SVM: 0.7905 ± 0.0154
Random Forest: 0.8237 ± 0.0100
MLP: 0.8074 ± 0.0184


In [60]:
# GridSearch 
param_grid = {
    "clf__C": [0.01, 0.1, 1, 10]
}

grid = GridSearchCV(
    pipelines["Logistic Regression"],
    param_grid=param_grid,
    cv=cv,
    scoring="f1_weighted",
    n_jobs=-1
)
grid.fit(X_train, y_train)

print("\nBest LR params:", grid.best_params_)
print("Best CV score:", grid.best_score_)

best_lr = grid.best_estimator_
y_pred_best = best_lr.predict(X_test)

print("\n=== Best Logistic Regression (GridSearch) Test ===")
print("Accuracy:", accuracy_score(y_test, y_pred_best))
print(classification_report(y_test, y_pred_best, zero_division=0))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_best))


Best LR params: {'clf__C': 0.01}
Best CV score: 0.8275763482761647

=== Best Logistic Regression (GridSearch) Test ===
Accuracy: 0.848
              precision    recall  f1-score   support

           0       0.82      0.82      0.82       211
           1       0.87      0.87      0.87       289

    accuracy                           0.85       500
   macro avg       0.84      0.84      0.84       500
weighted avg       0.85      0.85      0.85       500

Confusion Matrix:
 [[172  39]
 [ 37 252]]


In [61]:
joblib.dump(best_lr, "final_model.pkl")
print("\nFinal model kaydedildi final_model.pkl")




Final model kaydedildi final_model.pkl
